1. Apply some filters
- Choose only S&P 500 firms (SNP500=1)
- Sample period: From 2002 to 2010

In [49]:
import pandas as pd
import numpy as np

df = pd.read_excel("/content/merged_data.xlsx")

In [50]:
df_filtered = df[(df['SNP500'] == 1) & (df['FYEAR'] >= 2002) & (df['FYEAR'] <= 2010)]

2. Generate variables based on variable description
- If LossCarryForward is missing, replace it with 0.

In [51]:
df_clean = df_filtered.copy()
df_clean['TLCF'] = df_clean['TLCF'].fillna(0)

df_clean['BookLev'] = (df_clean['DLTT'] + df_clean['DLC']) / df_clean['AT']
df_clean['MktLev'] = (df_clean['DLTT'] + df_clean['DLC']) / (df_clean['AT'] - df_clean['CEQ'] + df_clean['MKVALT'] + df_clean['TXDB'])
df_clean['MB'] = (df_clean['AT'] + df_clean['MKVALT'] - (df_clean['SEQ'] - df_clean['PSTKL'] + df_clean['TXDITC'])) / df_clean['AT']
df_clean['FixedAsset'] = df_clean['PPENT'] / df_clean['AT']
df_clean['Profit'] = df_clean['EBIT'] / df_clean['AT']
df_clean['Size'] = np.log(df_clean['SALE'])
df_clean['TaxCredit'] = df_clean['TXDITC'] / df_clean['AT']
df_clean['LCF'] = df_clean['TLCF'] / df_clean['AT']

In [52]:
def winsorize(x, p=0.01):
  lower_bound = x.quantile(p)
  upper_bound = x.quantile(1 - p)
  return x.clip(lower=lower_bound, upper=upper_bound)

In [53]:
df_final = df_clean.copy()
v_winsorize = ['BookLev', 'MktLev', 'MB', 'FixedAsset', 'Profit', 'Size', 'Volatility', 'AbEarn', 'TaxCredit', 'LCF']
df_final[v_winsorize] = df_final[v_winsorize].apply(winsorize)

3. Generate summary statistics

In [54]:
summary_table = df_final[['BookLev', 'MktLev', 'MB', 'FixedAsset', 'Profit', 'Size', 'TaxCredit', 'LCF', 'Rating', 'Rated', 'InvGrade']].agg(
    ['mean', 'median', 'std', 'min', 'max']
).T.rename(columns={
    'mean': 'Mean',
    'median': 'Median',
    'std': 'Stdev',
    'min': 'Min',
    'max': 'Max'
}).round(2)

In [55]:
summary_table

,Mean,Median,Stdev,Min,Max
BookLev,0.26,0.24,0.16,0.00,0.74
MktLev,0.17,0.14,0.13,0.00,0.62
MB,1.81,1.52,0.93,0.80,5.67
FixedAsset,0.30,0.23,0.24,0.00,0.88
Profit,0.10,0.09,0.07,-0.10,0.35
Size,9.02,8.99,1.14,6.71,12.00
TaxCredit,0.04,0.02,0.05,0.00,0.24
LCF,0.03,0.00,0.09,0.00,0.59
Rating,8.04,8.00,2.80,1.00,18.00
Rated,0.91,1.00,0.29,0.00,1.00
